In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 适配 MNIST 数据集的简单卷积神经网络
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # x.shape = (b, 1, 28, 28)
        self.conv1 = nn.Sequential(
            # 卷积层，输入通道1，输出通道16，卷积核5x5，步长1，填充2
            nn.Conv2d(
                in_channels=1, 
                out_channels=16, 
                kernel_size=5, 
                stride=1, 
                padding=2, 
            ), 
            # x.shape = (b, 16, 28, 28)
            nn.ReLU(), 
            # 最大池化，核大小2，步长默认与核大小相同
            nn.MaxPool2d(kernel_size=2), 
            # x.shape = (b, 16, 14, 14)
        )

        self.conv2 = nn.Sequential(
            # 卷积层，输入通道16，输出通道32，卷积核5x5，步长1，填充2
            nn.Conv2d(
                in_channels=16, 
                out_channels=32, 
                kernel_size=5, 
                stride=1, 
                padding=2, 
            ), 
            # x.shape = (b, 32, 14, 14)
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2), 
            # x.shape = (b, 32, 7, 7)
        )
        
        # 用 mlp 做分类器
        self.classifier = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128), 
            nn.ReLU(), 
            nn.Linear(128, 10)
        )

    def forward(self, x):
        # x.shape = (b, 1, 28, 28)
        x = self.conv1(x)
        # x.shape = (b, 16, 14, 14)
        x = self.conv2(x)
        # x.shape = (b, 32, 7, 7)
        
        # 将特征展平为一维向量
        x = x.view(x.size(0), -1)
        # x.shape = (b, 32*7*7)

        # 通过分类器
        x = self.classifier(x)
        # x.shape = (b, 10)

        # 在使用nn.CrossEntropyLoss时，不需要在这里应用Softmax
        return x


In [2]:

# 计算准确率、精确率、召回率、F1分数
def metrics(all_targets, all_preds):
    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='macro')
    recall = recall_score(all_targets, all_preds, average='macro')
    f1 = f1_score(all_targets, all_preds, average='macro')
    return accuracy, precision, recall, f1

def train(model, device, train_loader, loss_func, optimizer, epoch):
    model.train()
    train_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"训练轮次: {epoch + 1} [{batch_idx * len(data)}/{len(train_loader.dataset)}] 损失: {loss.item():.6f}")
    
    print(f"Eposh {epoch + 1} 平均损失: {train_loss / len(train_loader):.6f}")

def test(model, device, test_loader, epoch):
    model.eval()
    all_targets, all_preds = [], []
    
    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            target = target.to(device)
            
            output = model(data)
            preds = output.argmax(dim=1)
            
            all_targets.extend(target.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy, precision, recall, f1 = metrics(all_targets, all_preds)
    print(f"Epoch {epoch + 1} 测试集: 准确率={accuracy:.4f}, 精确率={precision:.4f}, 召回率={recall:.4f}, F1分数={f1:.4f}\n")       

In [3]:

def main():
    device = 'cpu'
    if torch.cuda.is_available():
        device = 'cuda'
        print(f"Use GPU: {torch.cuda.get_device_name(0)}")
    elif torch.mps.is_available():
        device = 'mps'
        print("Use MPS")
    
    # 加载 MNIST 训练集
    # 参数：数据集的本地路径、使用训练集还是测试集、是否自动下载数据集、数据预处理流程
    train_dataset = datasets.MNIST(
        root='./data',
        train=True, 
        download=True, 
        transform=transforms.ToTensor()
    )
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

    # 加载 MNIST 测试集
    test_dataset = datasets.MNIST(
        root='./data',
        train=False,
        download=True,
        transform=transforms.ToTensor()
    )
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    model = SimpleCNN().to(device)
    
    # 分类问题，使用交叉熵损失函数
    loss_func = nn.CrossEntropyLoss()

    optimizer = optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(5):
        train(model, device, train_loader, loss_func, optimizer, epoch)
        test(model, device, test_loader, epoch)

if __name__ == '__main__':
    main()


训练轮次: 1 [0/60000] 损失: 2.302300
训练轮次: 1 [12800/60000] 损失: 2.253213
训练轮次: 1 [25600/60000] 损失: 2.140579
训练轮次: 1 [38400/60000] 损失: 1.488696
训练轮次: 1 [51200/60000] 损失: 0.754575
Eposh 1 平均损失: 1.669614
Epoch 1 测试集: 准确率=0.8405, 精确率=0.8591, 召回率=0.8377, F1分数=0.8403

训练轮次: 2 [0/60000] 损失: 0.597876
训练轮次: 2 [12800/60000] 损失: 0.507991
训练轮次: 2 [25600/60000] 损失: 0.307528
训练轮次: 2 [38400/60000] 损失: 0.390753
训练轮次: 2 [51200/60000] 损失: 0.324293
Eposh 2 平均损失: 0.403131
Epoch 2 测试集: 准确率=0.8821, 精确率=0.8994, 召回率=0.8804, F1分数=0.8824

训练轮次: 3 [0/60000] 损失: 0.392420
训练轮次: 3 [12800/60000] 损失: 0.287569
训练轮次: 3 [25600/60000] 损失: 0.258065
训练轮次: 3 [38400/60000] 损失: 0.314856
训练轮次: 3 [51200/60000] 损失: 0.155204
Eposh 3 平均损失: 0.275959
Epoch 3 测试集: 准确率=0.9311, 精确率=0.9328, 召回率=0.9305, F1分数=0.9306

训练轮次: 4 [0/60000] 损失: 0.206109
训练轮次: 4 [12800/60000] 损失: 0.213474
训练轮次: 4 [25600/60000] 损失: 0.149502
训练轮次: 4 [38400/60000] 损失: 0.234783
训练轮次: 4 [51200/60000] 损失: 0.406701
Eposh 4 平均损失: 0.213595
Epoch 4 测试集: 准确率=0.9443, 精确率=0.9458, 召